# F4·F5 PPN 시험 (notes/funnel_protocol.md, 규칙 notes/funnel_f4_ppn.md 2절)
런타임: **GPU(T4)**. 위에서부터 차례로 실행합니다. 결과는 Drive `data/funnel/ppn/`에 저장됩니다.

- 공식 코드 kshre/PPN @0210924, 파일 수정 없음(실행 래퍼만 사용).
- **성과 비노출:** 래퍼가 데이터의 앞 1133일(원래 학습 쪽)만 넘깁니다. 원래 시험 구간은 쓰지 않습니다. 가짜 시험 구간 지표도 파일로만 보내고 열지 않습니다(`smoke*.log`, `programlog`, `train_summary.csv`). 화면에는 설치 결과·시간·traceback만 냅니다.
- 실행 S(저장소 설정, 2001스텝, 상한 3시간)로 F4, 실행 P(배치 128, 2001스텝, 상한 4시간)로 F5를 잽니다. 이 노트북으로 재현 판정을 하지 않습니다.
- 측정은 1회입니다. 설치·실행 오류일 때만 다시 돌립니다.
- 미검증 노트북입니다. 오류가 나면 마지막 출력을 그대로 알려 주세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, time, json, subprocess
os.environ['MPLBACKEND'] = 'agg'
OUT = pathlib.Path('/content/drive/MyDrive/data/funnel/ppn'); OUT.mkdir(parents=True, exist_ok=True)
START = time.time(); (OUT/'log.txt').open('a').write(f'start {time.ctime()}\n')
print(subprocess.getoutput('nvidia-smi -L'))
assert 'Tesla T4' in subprocess.getoutput('nvidia-smi -L'), 'T4가 아닙니다. 사전 등록상 T4만 인정합니다. 런타임을 다시 받으세요.'

In [ ]:
%%bash
# 환경: Python 2.7(conda-forge) + CUDA 10.0/cuDNN 7.6.5(별도 prefix). 각 단계 로그는 /content/out/install_*.log
set -e
cd /content; mkdir -p /content/out
rm -rf /content/p27 /content/cu100
D=/content/drive/MyDrive/data/funnel/ppn
step(){ n=$1; shift; echo "== $n"; if ! "$@" > /content/out/install_$n.log 2>&1; then echo "실패: $n"; tail -25 /content/out/install_$n.log; cp /content/out/install_*.log "$D"/ 2>/dev/null; echo "로그를 Drive에 저장함"; exit 1; fi; tail -1 /content/out/install_$n.log; }
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
step py27 ./bin/micromamba create -y -p /content/p27 -c conda-forge python=2.7 pip numpy=1.16 pandas=0.24 scipy=1.2 matplotlib=2.2 seaborn=0.9 cvxopt
step cuda bash -c "./bin/micromamba create -y -p /content/cu100 -c conda-forge cudatoolkit=10.0 cudnn=7.6.5 || { rm -rf /content/cu100; ./bin/micromamba create -y -p /content/cu100 -c https://repo.anaconda.com/pkgs/main cudatoolkit=10.0 cudnn=7.6.5; }"
P="/content/p27/bin/python -m pip --default-timeout=120 --retries 5"
# grpcio 등은 py2.7 휠이 있는 버전으로 고정(최신 sdist 소스 빌드 실패 대응, 11-06)
step tfdeps $P install -q --prefer-binary --only-binary=grpcio,h5py grpcio==1.27.2 h5py==2.10.0 protobuf==3.17.3
step tf $P install -q --prefer-binary --only-binary=grpcio,h5py tensorflow-gpu==1.15.0 tflearn==0.3.2 pympler==0.8
[ -d PPN ] || git clone -q https://github.com/kshre/PPN.git
cd PPN && git checkout -q 0210924 && git log -1 --format='%h %ad' --date=short

In [ ]:
%%bash
# 관문: 버전, GPU 인식, import (실패하면 이 칸이 오류로 끝남)
set -eo pipefail
export LD_LIBRARY_PATH=/content/cu100/lib:$LD_LIBRARY_PATH MPLBACKEND=agg TF_CPP_MIN_LOG_LEVEL=2
cd /content/PPN
/content/p27/bin/python - <<'PY' 2>&1 | tee /content/out/install_versions.log
# -*- coding: utf-8 -*-
import sys, numpy, pandas, tensorflow as tf, tflearn
print('python', sys.version.split()[0]); print('numpy', numpy.__version__, 'pandas', pandas.__version__)
print('tf', tf.__version__, 'tflearn', getattr(tflearn, '__version__', '?'))
g = tf.test.is_gpu_available(); print('gpu', g)
import main  # import check only (py2: ASCII only here)
print('import main ok')
assert sys.version_info[:2] == (2, 7) and tf.__version__ == '1.15.0'
if not g: sys.exit('GPU not visible to TF')
PY
cp /content/out/install_*.log /content/drive/MyDrive/data/funnel/ppn/ 2>/dev/null; true

위 출력에서 `python 2.7…`, `tf 1.15.0`, `gpu True`, `import main ok`를 확인합니다. `gpu False`면 그대로 알려 주세요(사전 등록 2.2: 환경을 고치고, 끝내 안 되면 CPU로 측정).

In [ ]:
# 데이터: README의 Drive database.zip(저자 공유). 크기·해시·목록만 확인합니다.
import hashlib, zipfile, shutil, glob
z = '/content/database.zip'
if not os.path.exists(z):
    subprocess.run(['gdown', '1_HmTwuJky-al7jDQYpBGcPPj3o6F85Yf', '-O', z], check=True)
h = hashlib.sha256(open(z,'rb').read()).hexdigest()
zf = zipfile.ZipFile(z); names = [(i.filename, i.file_size) for i in zf.infolist()]
print('size', os.path.getsize(z), 'sha256', h); print(names)
shutil.rmtree('/content/dbz', ignore_errors=True); zf.extractall('/content/dbz')
os.makedirs('/content/PPN/database', exist_ok=True)
for fn in ['all_stocks_5yr.csv', 'data_new.pkl']:
    hits = [p for p in glob.glob('/content/dbz/**/'+fn, recursive=True) if '__MACOSX' not in p]
    print(fn, hits)
    if hits: shutil.copy(hits[0], '/content/PPN/database/'+fn)
assert os.path.exists('/content/PPN/database/all_stocks_5yr.csv'), 'all_stocks_5yr.csv 없음'
import pandas as pd
df = pd.read_csv('/content/PPN/database/all_stocks_5yr.csv', usecols=['date','Name'])
d = sorted(df['date'].unique())
info = {'zip_size': os.path.getsize(z), 'zip_sha256': h, 'zip_names': names, 'n_names': int(df['Name'].nunique()), 'n_dates': len(d), 'first': d[0], 'last': d[-1],
        'kept_last_date(index 1132)': d[1132], 'original_test_first_date(index 1133)': d[1133], 'pkl': os.path.exists('/content/PPN/database/data_new.pkl')}
print(json.dumps(info, indent=1)); (OUT/'data_info.json').write_text(json.dumps(info, indent=1))
assert info['n_names'] == 505 and info['n_dates'] == 1259, '데이터 규격이 코드 가정(505종목, 1259일)과 다릅니다'
LAST_KEPT = d[1132]
# 데이터 원천 규칙(사전 등록 2.1): pkl이 있으면 pkl, py2.7 pandas로 못 읽으면 지우고 코드가 CSV로 다시 만든다
pk = '/content/PPN/database/data_new.pkl'
if os.path.exists(pk):
    chk = subprocess.run(['/content/p27/bin/python','-c',"import pandas as pd; p=pd.read_pickle('%s'); print(type(p).__name__, p.values.shape)" % pk], capture_output=True, text=True)
    print('pkl check:', chk.returncode, chk.stdout.strip(), chk.stderr.strip()[-300:])
    if chk.returncode != 0:
        os.remove(pk); print('pkl 로드 실패 → 삭제, 코드가 CSV에서 다시 만듦')
    info['pkl_load_ok'] = chk.returncode == 0
    (OUT/'data_info.json').write_text(json.dumps(info, indent=1))

In [ ]:
# 실행 래퍼(허용 수정: 장치 지정, 입력 자르기, 결과 기록). 공식 파일은 수정하지 않는다. Python 2.7용.
WRAP = r'''# -*- coding: utf-8 -*-
from __future__ import print_function
import os, sys, time, json, traceback
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
sys.path.insert(0, os.getcwd())
CUT = 1133
TIM = os.environ['PPN_TIMES']; ERR = os.environ['PPN_ERR']
def mark(ev, **kw):
    d = {'ev': ev, 't': time.time(), 'pid': os.getpid()}; d.update(kw)
    with open(TIM, 'a') as f: f.write(json.dumps(d) + '\n')
def err():
    with open(ERR, 'a') as f: traceback.print_exc(file=f)

import pgportfolio.marketdata.globaldatamatrix as gdm
_gp = gdm.HistoryManager.get_global_panel
LAST = os.environ['PPN_LAST_KEPT']  # 정렬된 전체 날짜의 1133번째(0부터 1132), 노트북이 CSV에서 계산
def cut_panel(self, *a, **k):
    src = 'pkl' if os.path.exists('./database/data_new.pkl') else 'csv'
    t = time.time(); p = _gp(self, *a, **k)
    idx = [str(x)[:10] for x in p.minor_axis]
    n = len(idx)
    if n != 1259: raise RuntimeError('panel length %d != 1259' % n)
    if idx != sorted(idx): raise RuntimeError('panel dates not sorted')
    keep = [x for x, s in zip(p.minor_axis, idx) if s <= LAST]
    if len(keep) != CUT: raise RuntimeError('kept %d != %d' % (len(keep), CUT))
    q = p.reindex(minor_axis=keep)
    mark('panel', source=src, n_full=n, n_kept=len(q.minor_axis), last_kept=str(q.minor_axis[-1]),
         dtype=str(q.values.dtype), shape=list(q.values.shape), load_sec=time.time() - t)
    return q
gdm.HistoryManager.get_global_panel = cut_panel

import pgportfolio.learn.tradertrainer as tt
_lbs = tt.TraderTrainer.log_between_steps
def lbs(self, step):
    mark('step', step=step); return _lbs(self, step)
tt.TraderTrainer.log_between_steps = lbs

import pgportfolio.trade.backtest as bt
_st = bt.BackTest.start_trading
def st(self, *a, **k):
    mark('bt_start', n_test=int(self._total_steps)); r = _st(self, *a, **k); mark('bt_end'); return r
bt.BackTest.start_trading = st

import pgportfolio.autotrain.training as tr
_t1 = tr.train_one
def train_one(*a, **k):
    mark('train_one_start')
    try:
        r = _t1(*a, **k); mark('train_one_end'); return r
    except Exception:
        err(); mark('train_one_error'); raise
tr.train_one = train_one

import main as ppn_main
sys.argv = ['main.py', '--mode=train', '--processes=1', '--device=' + os.environ.get('PPN_DEVICE', 'gpu')]
mark('main_start')
try:
    ppn_main.main()
except Exception:
    err(); mark('main_error'); raise
mark('main_end')
'''
pathlib.Path('/content/out/run_ppn.py').write_text(WRAP)
print('wrapper ok')

In [ ]:
import resource, signal
DEVICE = 'gpu'  # 사전 등록 2.2: GPU를 끝내 못 쓰면 'cpu'로 바꾸고 그 사실을 기록
CAPS = {'S': 3*3600, 'P': 4*3600}  # 실행별 벽시계 상한(사전 등록 2.4)

def prepare(name, batch):
    w = f'/content/run_{name}'
    shutil.rmtree(w, ignore_errors=True)
    shutil.copytree('/content/PPN', w, ignore=shutil.ignore_patterns('.git', 'database'))
    os.symlink('/content/PPN/database', w+'/database')
    cfgp = w+'/train_package/1/net_config.json'
    cfg = json.load(open(cfgp))
    before = json.dumps(cfg['training'], sort_keys=True)
    cfg['training']['steps'] = 2001          # 허용 수정 4: 실행 길이
    if batch: cfg['training']['batch_size'] = batch  # P만: 논문 배치
    json.dump(cfg, open(cfgp, 'w'), indent=4)
    return w, {'training_before': before, 'training_after': json.dumps(cfg['training'], sort_keys=True)}

def run(name, batch=None):
    w, diff = prepare(name, batch)
    times, errp = f'/content/out/{name}_times.jsonl', f'/content/out/{name}_error.log'
    for p in (times, errp): pathlib.Path(p).unlink(missing_ok=True)
    env = dict(os.environ, PPN_LAST_KEPT=LAST_KEPT, PPN_TIMES=times, PPN_ERR=errp, PPN_DEVICE=DEVICE, MPLBACKEND='agg', TF_CPP_MIN_LOG_LEVEL='2',
               LD_LIBRARY_PATH='/content/cu100/lib:'+os.environ.get('LD_LIBRARY_PATH',''))
    gpu_f = open(f'/content/out/{name}_gpu.csv', 'w')
    sampler = subprocess.Popen(['nvidia-smi','--query-gpu=utilization.gpu,memory.used','--format=csv,noheader,nounits','-l','1'], stdout=gpu_f)
    cpu0 = resource.getrusage(resource.RUSAGE_CHILDREN); t0 = time.time(); killed = False
    with open(f'/content/out/{name}_smoke.log','w') as f, open(f'/content/out/{name}_smoke_err.log','w') as g:
        p = subprocess.Popen(['/content/p27/bin/python', '/content/out/run_ppn.py'], cwd=w, stdout=f, stderr=g, env=env, start_new_session=True)
        while p.poll() is None:
            time.sleep(10)
            if time.time()-t0 > CAPS[name]:
                os.killpg(p.pid, signal.SIGKILL); killed = True; p.wait(); break
    rc = p.returncode; sec = time.time()-t0
    cpu1 = resource.getrusage(resource.RUSAGE_CHILDREN)
    sampler.terminate(); sampler.wait(); gpu_f.close()
    smp = [[float(x) for x in l.split(',')] for l in open(f'/content/out/{name}_gpu.csv') if l.count(',') == 1 and '[' not in l]
    ev = [json.loads(l) for l in open(times)] if os.path.exists(times) else []
    first = lambda e, **kw: next((x for x in ev if x['ev'] == e and all(x.get(k) == v for k, v in kw.items())), None)
    m0, m1, m2 = first('step', step=0), first('step', step=1000), first('step', step=2000)
    bs, be, te = first('bt_start'), first('bt_end'), first('train_one_end')
    summary = os.path.exists(w+'/train_package/train_summary.csv')  # 존재만 확인, 열지 않음
    r = {'run': name, 'batch': batch or 'repo(32)', 'rc': rc, 'killed_at_cap': killed, 'seconds': sec,
         'f4_pass': bool(rc == 0 and te and be and summary), 'train_summary_exists': summary,
         'startup': (m0['t']-t0) if m0 else None, 'block': (m2['t']-m1['t']) if (m1 and m2) else None,
         'first_block': (m1['t']-m0['t']) if (m0 and m1) else None,
         'tail': (bs['t']-m2['t']) if (bs and m2) else None, 'bt': (be['t']-bs['t']) if (bs and be) else None,
         'bt_steps': bs.get('n_test') if bs else None, 'panel': first('panel'),
         'events': [x['ev'] for x in ev],
         'cpu_seconds_per_wall_second': (cpu1.ru_utime+cpu1.ru_stime-cpu0.ru_utime-cpu0.ru_stime)/sec, 'cpu_count': os.cpu_count(),
         'gpu_util_mean': (sum(s[0] for s in smp)/len(smp) if smp else None), 'gpu_mem_mb_max': (max(s[1] for s in smp) if smp else None),
         'device': DEVICE, 'gpu': subprocess.getoutput('nvidia-smi -L'), 'config_diff': diff}
    k = 100 if batch == 128 else 10  # P: 논문 10^5스텝, S: 저장소 10^4스텝
    if None not in (r['startup'], r['block'], r['tail'], r['bt']) and r['bt_steps']:
        r['hours_converted'] = (r['startup'] + k*r['block'] + r['tail'] + r['bt']*94/r['bt_steps'])/3600
    else:
        r['hours_converted'] = None
    # 하한(사전 등록 2.5): 백테스트 전에 끊기거나 bt가 없으면 학습 부분만으로 계산
    blk = r['block'] or r['first_block']
    if r['startup'] is not None and blk is not None:
        r['hours_lower_bound'] = (r['startup'] + k*blk + (r['tail'] or 0))/3600
    elif killed and m1 is None:
        r['hours_lower_bound'] = k*CAPS[name]/3600  # 1000스텝도 상한 안에 못 끝남
    else:
        r['hours_lower_bound'] = None
    # 화면용 오류 요약: 사전 고정 필터(성과 문구가 든 줄은 제외)
    import re as _re
    se = f'/content/out/{name}_smoke_err.log'
    if os.path.exists(se):
        keep = [l.rstrip()[:200] for l in open(se, errors='ignore')
                if _re.search(r'Traceback|Error|Killed|CUDA|OOM|Segmentation|Aborted', l)
                and not _re.search(r'portfolio|log.?mean|loss|pv|value|benefit', l, _re.I)]
        r['stderr_filtered_tail'] = keep[-15:]
    for n in [f'{name}_times.jsonl', f'{name}_error.log', f'{name}_gpu.csv', f'{name}_smoke.log', f'{name}_smoke_err.log']:
        if os.path.exists('/content/out/'+n): shutil.copy('/content/out/'+n, OUT/n)
    (OUT/f'f4_{name}.json').write_text(json.dumps(r, indent=1))
    print(json.dumps({k2: v for k2, v in r.items() if k2 != 'config_diff'}, indent=1))
    if os.path.exists(errp): print('--- traceback(마지막 30줄) ---'); print(''.join(open(errp).readlines()[-30:]))
    return r

rS = run('S')

In [ ]:
# S가 오류로 실패했으면 멈춥니다. 상한으로 끊긴 것은 실패가 아니므로 P를 돌립니다(사전 등록 2.5).
assert rS['f4_pass'] or rS['killed_at_cap'], 'S가 오류로 실패했습니다. 위 traceback을 알려 주세요.'
rP = run('P', batch=128)
A, A_lb, B = rP['hours_converted'], rP['hours_lower_bound'], rS['hours_converted']
if A is not None:
    f5 = '한도 초과' if A > 10 else '통과(사용자 보고 후 F7 사전 등록)'
elif A_lb is not None and A_lb > 10:
    f5 = '한도 초과(하한 기준)'
else:
    f5 = 'F5 측정 실패'
f4 = '통과(가짜 시험 구간 기준, 원 시험 구간 미실행)' if rS['f4_pass'] else ('상한 내 미완' if rS['killed_at_cap'] else '실패')
res = {'F4': f4, 'A_hours_paper_procedure': A, 'A_lower_bound': A_lb, 'B_hours_repo_config(참고)': B,
       'B_lower_bound(참고)': rS['hours_lower_bound'], 'device': DEVICE, 'F5': f5}
print(json.dumps(res, indent=1, ensure_ascii=False)); (OUT/'f5_result.json').write_text(json.dumps(res, indent=1, ensure_ascii=False))
print('작업 시간(시간):', round((time.time()-START)/3600, 2))
(OUT/'log.txt').open('a').write(f'end {time.ctime()}\n')